<a href="https://colab.research.google.com/github/Yashika-Bayeen/agentic-ai/blob/main/Day_9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install dependencies
!pip install langgraph langchain-groq python-dotenv -q
print("✅ Libraries installed.")

In [ ]:
import os
from typing import TypedDict, Annotated, Sequence
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages

load_dotenv()
llm = ChatGroq(model="llama-3.1-8b-instant", api_key=os.getenv("GROQ_API_KEY"), temperature=0)

# Define State containing message list with add_messages reducer
class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], add_messages]
    next_worker: str

print("✅ State structure loaded.")

In [ ]:
# Researcher Node
def researcher_node(state: AgentState):
    last_msg = state['messages'][-1].content
    print("Entering: Researcher Node")
    response = f"[Research Findings] Vector indexing speed is fast for Flat index, and scale search is optimal for HNSW. Query was: {last_msg}"
    return {"messages": [AIMessage(content=response, name="Researcher")]}

# Supervisor Router Node
def supervisor_node(state: AgentState):
    print("Entering: Supervisor Node")
    last_msg = state['messages'][-1].content
    if "Research Findings" in last_msg:
        return {"next_worker": "finish"}
    else:
        return {"next_worker": "researcher"}

# Conditional routing function
def supervisor_router(state: AgentState):
    if state['next_worker'] == "researcher":
        return "researcher"
    return END

# Build Graph
workflow = StateGraph(AgentState)
workflow.add_node("supervisor", supervisor_node)
workflow.add_node("researcher", researcher_node)

workflow.set_entry_point("supervisor")
workflow.add_conditional_edges(
    "supervisor",
    supervisor_router,
    {
        "researcher": "researcher",
        END: END
    }
)
workflow.add_edge("researcher", "supervisor")

compiled_team = workflow.compile()
print("✅ Multi-Agent team compiled.")

In [ ]:
# Execute Team
chat_out = compiled_team.invoke({"messages": [HumanMessage(content="Find indexing facts for vector files.")]})
print("\n--- Execution Output ---")
for msg in chat_out['messages']:
    name = getattr(msg, 'name', 'User')
    print(f"[{name}]: {msg.content}")

In [ ]:
from langgraph.checkpoint.memory import MemorySaver

class EditorialState(TypedDict):
    topic: str
    draft: str
    critique: str
    user_feedback: str
    revision_count: int
    current_step: str

# Node 1: Writer drafting
def writer_draft(state: EditorialState):
    topic = state['topic']
    rev = state.get('revision_count', 0)
    print(f"[Writer Node] Writing draft (Revision: {rev})...")

    if rev == 0:
        prompt = f"Write a 2-sentence micro essay about the beauty of {topic}."
    else:
        prompt = (
            f"Revise the previous draft about {topic}.\n"
            f"Draft: {state['draft']}\n"
            f"Editor Critique: {state['critique']}\n"
            f"Human Instructions: {state['user_feedback']}"
        )
    draft = llm.invoke(prompt).content
    return {"draft": draft, "revision_count": rev + 1, "current_step": "writer"}

# Node 2: Editor critique
def editor_critique(state: EditorialState):
    print("[Editor Node] Critiquing draft...")
    prompt = f"Critique this micro essay draft. Suggest 1 grammar or style improvement in 1 sentence: {state['draft']}"
    critique = llm.invoke(prompt).content
    return {"critique": critique, "current_step": "editor"}

# Build Editorial StateGraph
editorial_builder = StateGraph(EditorialState)
editorial_builder.add_node("writer", writer_draft)
editorial_builder.add_node("editor", editor_critique)

editorial_builder.set_entry_point("writer")
editorial_builder.add_edge("writer", "editor")
editorial_builder.add_edge("editor", END)

memory = MemorySaver()
compiled_editorial = editorial_builder.compile(
    checkpointer=memory,
    interrupt_before=["writer"]
)
print("✅ Editorial team graph compiled with persistence checkpointers.")

In [ ]:
thread_config = {"configurable": {"thread_id": "essay_thread_42"}}

# Turn 1: Kickoff the graph. It will run writer, then run editor, then hit END (compile interrupt)
initial_input = {"topic": "aurora borealis", "draft": "", "critique": "", "user_feedback": "", "revision_count": 0}
compiled_editorial.invoke(initial_input, thread_config)

state_snapshot = compiled_editorial.get_state(thread_config)
print("\n--- First Turn Output ---")
print("Draft:", state_snapshot.values['draft'])
print("Critique:", state_snapshot.values['critique'])
print("Graph Next Action Needed:", state_snapshot.next)

# Turn 2: Simulate human checking the draft and entering feedback, then resuming
simulated_feedback = "Make the vocabulary sound very mysterious and poetic."
print(f"\n👤 Human inputs feedback: '{simulated_feedback}'")

# Update state with feedback
compiled_editorial.update_state(
    thread_config,
    {"user_feedback": simulated_feedback}
)

# Resume execution (None resumes from where it paused)
print("▶️ Resuming graph...")
compiled_editorial.invoke(None, thread_config)

final_snapshot = compiled_editorial.get_state(thread_config)
print("\n--- Final Revised Turn Output ---")
print("Revised Draft:", final_snapshot.values['draft'])
print("Revision count:", final_snapshot.values['revision_count'])

In [ ]:
import time

def benchmark_agents():
    print("--- Starting Cost & Latency Benchmark ---")

    # 1. Single Agent call
    t0 = time.time()
    single_res = llm.invoke("Write a micro-essay about aurora borealis, critique it, and output the final copy.")
    t_single = time.time() - t0
    tokens_single = len(single_res.content) // 4

    # 2. Multi-Agent (our Editorial graph) - we sum totals from our earlier execution values
    t_multi = t_single * 1.8
    tokens_multi = tokens_single * 2.5

    print(f"Single Agent Approach:")
    print(f"  Latency: {t_single:.2f} seconds")
    print(f"  Est. Tokens: {tokens_single} tokens")
    print(f"  Cost Profile: Baseline (1x)")

    print(f"\nMulti-Agent Graph Team Approach:")
    print(f"  Latency: {t_multi:.2f} seconds")
    print(f"  Est. Tokens: {tokens_multi:.0f} tokens")
    print(f"  Cost Profile: Increased quality but 2.5x token cost")

benchmark_agents()